# 04 — Machine Learning Model Experiment
**SmartBiz AI V1.0 — Phase 4: ML Forecasting Models**

This notebook trains and evaluates advanced machine learning models for demand forecasting:
1. Random Forest
2. XGBoost
3. LightGBM

We compare these models against each other and extract feature importances.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
import json
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print('Libraries loaded.')

## 1. Load Preprocessed Data

In [ ]:
from src.data.loader import load_csv
from src.preprocessing.cleaning import clean_sales_data
from src.preprocessing.transformation import filter_product, split_train_test
from src.preprocessing.feature_engineering import engineer_features

# Load and run full preprocessing for Product 1
df = load_csv('../../data/raw/sample_sales_data.csv')
df = clean_sales_data(df)
df_p1 = filter_product(df, product_id=1)
df_p1 = engineer_features(df_p1)

train_df, test_df = split_train_test(df_p1, test_ratio=0.2)
print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')

## 2. Run ML Training Pipeline

In [ ]:
from src.forecasting.training_pipeline import run_ml_experiment

predictions, importances, y_test = run_ml_experiment(train_df, test_df)

for name in predictions.keys():
    print(f'\n{name} Feature Importances:')
    print(importances[name].head(3))

## 3. Evaluate Models

In [ ]:
from src.forecasting.evaluation import evaluate_all_models

eval_results = evaluate_all_models(y_test, predictions)
results_df = pd.DataFrame(eval_results)
results_df

## 4. Visualize Predictions

In [ ]:
# Drop NaNs from test_df to align with y_test
test_dates = test_df.dropna()['date'].values

plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test, 'ko-', label='Actual', linewidth=2)

colors = ['#e74c3c', '#3498db', '#2ecc71']
for (name, preds), color in zip(predictions.items(), colors):
    plt.plot(test_dates, preds, '--', label=name, linewidth=1.5, color=color)

plt.title('ML Model Predictions vs Actual (Product 1)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales Qty')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../results/plots/ml_predictions.png', dpi=150)
plt.show()

## 5. Feature Importance Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, imp_df) in zip(axes, importances.items()):
    ax.barh(imp_df['feature'], imp_df['importance'], color='teal', edgecolor='black')
    ax.invert_yaxis()  # Highest importance at the top
    ax.set_title(f'{name} Feature Importance')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('../../results/plots/feature_importances.png', dpi=150)
plt.show()

## 6. Save Experiment Outputs

In [ ]:
from src.forecasting.training_pipeline import save_experiment_results

save_experiment_results(predictions, importances, test_dates, y_test, results_dir='../../results')

results_df.to_csv('../../results/metrics/ml_metrics.csv', index=False)
print('Metrics table saved.')

## Summary
- Successfully trained **Random Forest**, **XGBoost**, and **LightGBM**.
- Models successfully integrated with Phase 3 preprocessing.
- Evaluated using Phase 3 metrics (MAE, RMSE, WAPE, sMAPE).
- Generated predictions, extracted feature importances, and saved results to disk.